# Lección 11.1 — Datos con orden: texto, audio y series de tiempo

> **Módulo 11 · Secuencias y lenguaje** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.1 (tipos de datos), Lección 2.6 (fuga de datos), Lección 4.6 (validación cruzada temporal), Lección 4.7 (líneas base)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué tienen en común el texto, el audio y las series de tiempo: **el orden importa**.
- Convertir una secuencia en una tabla de ejemplos con **ventanas deslizantes**.
- Construir y vencer (o no) a las **líneas base ingenuas** de una serie de tiempo.
- Evitar el error más caro con secuencias: **mezclar el pasado con el futuro** al validar.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

plt.rcParams["figure.figsize"] = (9, 3.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el orden también es información

Lee estos dos titulares:

```
🗞️  "El perro muerde al hombre"     → noticia aburrida
🗞️  "El hombre muerde al perro"     → ¡noticia de portada!
```

Tienen **exactamente las mismas palabras**. Lo único que cambia es el **orden**… y el significado se da la vuelta.

Hasta ahora casi todos nuestros datos eran **tablas**: cada fila (un cliente, una casa) era independiente de las demás y el orden de las columnas daba igual. Pero muchos datos del mundo real son **secuencias**: una cosa *después* de otra.

| Secuencia | Cada "paso" es… | Ejemplo |
|---|---|---|
| 📝 Texto | una palabra (o un pedazo de palabra) | "El hombre muerde al perro" |
| 🎵 Audio | la presión del aire en un instante | 16,000 números por segundo |
| 📈 Serie de tiempo | una medición en una fecha/hora | ventas diarias, temperatura por hora |
| 🧬 ADN | una letra | A, C, G, T… |
| 🖱️ Clics de un usuario | una acción | inicio → búsqueda → carrito → pago |

En todas, **si barajas los pasos, destruyes la información**.

### 🔮 Ejercicio 1 — ¿Lo nota el modelo?
En el Módulo 5 representábamos un texto contando cuántas veces aparece cada palabra (la "bolsa de palabras", *bag of words*). Vamos a pasar los dos titulares por esa representación.

¿Saldrán dos vectores **distintos** o **idénticos**?

✍️ *Mi predicción:*

In [ ]:
titulares = ["El perro muerde al hombre", "El hombre muerde al perro"]

bolsa = CountVectorizer()
vectores = bolsa.fit_transform(titulares).toarray()
pd.DataFrame(vectores, columns=bolsa.get_feature_names_out(), index=["Titular 1", "Titular 2"])

<details><summary><b>👀 Ver respuesta</b></summary>

**Idénticos.** Cada palabra aparece una vez en cada titular, así que la bolsa de palabras ve lo mismo. Para un modelo que solo recibe estos números, *"el perro muerde al hombre"* y *"el hombre muerde al perro"* son **la misma frase**.

Moraleja: si el orden importa, necesitamos representaciones y modelos que **respeten el orden**. De eso trata todo este módulo.
</details>

---
## 2. 📖 El concepto: tres sabores de secuencia

Una **secuencia** es una lista de pasos donde **la posición importa**: x₁, x₂, x₃, …, xₜ. El subíndice *t* se suele llamar "tiempo", aunque en un texto sea la posición de la palabra.

Veamos los tres sabores más comunes, generados aquí mismo:

In [ ]:
# 📝 Texto: una lista de palabras (tokens; los veremos a fondo en la Lección 11.2)
frase = "el hombre muerde al perro".split()

# 🎵 Audio: una nota "la" (440 vibraciones por segundo) grabada a 16,000 muestras por segundo
muestras_por_segundo = 16_000
tiempo_audio = np.arange(0, 0.01, 1 / muestras_por_segundo)          # solo 0.01 segundos
nota = 0.6 * np.sin(2 * np.pi * 440 * tiempo_audio) + 0.3 * np.sin(2 * np.pi * 880 * tiempo_audio)

# 📈 Serie de tiempo: ventas diarias de una tienda durante 2 años
rng = np.random.default_rng(42)
dias = pd.date_range("2023-01-01", periods=730, freq="D")
t = np.arange(730)
ventas = (200 + 0.3 * t                                                  # tendencia: la tienda crece
          + np.array([0, -5, -3, 2, 10, 35, 20])[dias.day_of_week]      # patrón semanal (lun..dom)
          + 30 * np.sin(2 * np.pi * t / 365.25)                          # patrón anual
          + rng.normal(0, 8, 730))                                       # ruido del día a día
serie = pd.Series(ventas.round(1), index=dias, name="ventas")

print("Texto :", frase, f"→ {len(frase)} pasos")
print(f"Audio : {len(nota)} números para 0.01 s de sonido")
print(f"Serie : {len(serie)} días, de {serie.index[0].date()} a {serie.index[-1].date()}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(tiempo_audio * 1000, nota, ".-"); ax[0].set_title("Audio: 0.01 s de una nota 'la'"); ax[0].set_xlabel("milisegundos")
ax[1].plot(serie); ax[1].set_title("Ventas diarias"); ax[1].set_ylabel("unidades")
plt.tight_layout(); plt.show()

### ✋ Ejercicio 2 — Haz cuentas con secuencias
1. Una canción de **3 minutos** grabada a 16,000 muestras por segundo, ¿cuántos números tiene?
2. En la gráfica de ventas, ¿qué **tres patrones** distingues a simple vista? (Pista: mira la tendencia general, la "ola" grande y el "serrucho" fino.)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. 3 min = 180 s → 180 × 16,000 = **2,880,000 números**. Las secuencias de audio son larguísimas; por eso los modelos de voz suelen resumir el audio en trozos antes de procesarlo.
2. Los tres patrones con los que generamos la serie:
   - **Tendencia:** la tienda vende cada vez más (de ~200 a ~420 unidades).
   - **Estacionalidad anual:** la "ola" que sube y baja a lo largo del año.
   - **Estacionalidad semanal:** el "serrucho" fino; los viernes, sábados y domingos se vende más.

   Y encima de todo, **ruido**: lo que ningún modelo podrá predecir.
</details>

### 📖 Lo que hace especiales a las secuencias
1. **Dependencia:** el paso *t* se parece al paso *t − 1* (las ventas de hoy se parecen a las de ayer). En una tabla normal, la fila 5 no tiene nada que ver con la fila 6.
2. **Longitud variable:** una frase puede tener 3 palabras o 300. Una tabla siempre tiene las mismas columnas.
3. **Dirección:** en una serie de tiempo **el futuro no puede usarse para predecir el pasado**. En texto, en cambio, a veces sí se lee en ambas direcciones (lo verás con BERT en la Lección 12.4).

### 🔮 Ejercicio 3 — ¿Secuencia o no?
¿En cuáles de estos casos el orden de los datos **importa**?
- a) Fotos de 1,000 gatos distintos para un clasificador de razas
- b) Los latidos de un electrocardiograma
- c) Una encuesta de satisfacción con 10 preguntas numéricas
- d) Los movimientos de una tarjeta de crédito para detectar fraude

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **a) No.** Cada foto es independiente: puedes barajarlas sin perder nada.
- **b) Sí.** Un latido raro se reconoce por su forma *a lo largo del tiempo*.
- **c) No, en general.** Las columnas tienen nombre fijo; cambiar el orden de las columnas no cambia la información. Es una tabla.
- **d) Sí.** "Compra de $5 en una gasolinera y, 3 minutos después, $2,000 en electrónica en otro país" es sospechoso **por el orden y el tiempo** entre movimientos.
</details>

---
## 3. 📖 Cómo se ve una secuencia por dentro: ventanas deslizantes

Los modelos del Módulo 5 necesitan una **tabla**: filas de ejemplos con columnas fijas. ¿Cómo convertimos una serie larga en una tabla?

Con una **ventana deslizante**: miramos los últimos *n* pasos y pedimos predecir el siguiente. Luego corremos la ventana un paso y repetimos.

```
Serie:   10  12  13  15  16  18
         └──┬──┘  ↓
         [10 12 13] → 15        ← ejemplo 1
             [12 13 15] → 16    ← ejemplo 2
                 [13 15 16] → 18 ← ejemplo 3
```

Cada ventana se vuelve una **fila**; los *n* valores pasados son las características (se suelen llamar *lags*, "retrasos") y el valor siguiente es la etiqueta. ¡Así cualquier modelo de tabla puede trabajar con secuencias!

In [ ]:
def crear_ventanas(secuencia, tamano):
    # Convierte una secuencia en una tabla: cada fila = los últimos `tamano` valores → el siguiente
    v = np.asarray(secuencia)
    X = np.array([v[i:i + tamano] for i in range(len(v) - tamano)])
    y = v[tamano:]
    return X, y

X_mini, y_mini = crear_ventanas([10, 12, 13, 15, 16, 18], tamano=3)
tabla = pd.DataFrame(X_mini, columns=["hace 3", "hace 2", "hace 1"])
tabla["→ siguiente"] = y_mini
tabla

### 🔮 Ejercicio 4 — Cuenta las filas
Tenemos una serie de **730 días**. Si usamos ventanas de tamaño **7**, ¿cuántas filas tendrá la tabla? ¿Y con ventanas de tamaño **28**?

✍️ *Mis predicciones:*

In [ ]:
for tamano in [7, 28]:
    X_v, y_v = crear_ventanas(serie, tamano)
    print(f"Ventana de {tamano:>2} días → tabla de {X_v.shape[0]} filas × {X_v.shape[1]} columnas")

<details><summary><b>👀 Ver respuesta</b></summary>

**723** filas con ventana 7 y **702** con ventana 28: siempre `730 − tamaño`. Los primeros días no tienen suficiente "pasado" para llenar una ventana, así que se pierden. Ventanas más grandes = más memoria por ejemplo, pero menos ejemplos.
</details>

---
## 4. 📖 Antes de cualquier modelo: las líneas base ingenuas

En la Lección 4.7 aprendiste a preguntar: *"¿le gano a adivinar el promedio?"*. Con series de tiempo hay líneas base mucho más duras de vencer:

| Línea base | Regla | Funciona bien cuando… |
|---|---|---|
| **Ingenua** (*naive*) | "Mañana = hoy" | La serie cambia lento |
| **Estacional ingenua** | "El próximo sábado = el sábado pasado" | Hay un ciclo fuerte (semanal, diario…) |
| **Promedio móvil** | "Mañana = promedio de los últimos *k* días" | Hay mucho ruido |

Parecen tontas, pero **muchos modelos sofisticados no les ganan**. Si tu modelo no supera a "mañana = hoy", no sirve.

Primero separamos el pasado (80 %) del futuro (20 %) **respetando el orden**: entrenamos con los primeros días y evaluamos con los últimos.

In [ ]:
corte = int(len(serie) * 0.8)
entreno, prueba = serie[:corte], serie[corte:]
print(f"Entrenamiento: {entreno.index[0].date()} → {entreno.index[-1].date()}  ({len(entreno)} días)")
print(f"Prueba       : {prueba.index[0].date()} → {prueba.index[-1].date()}  ({len(prueba)} días)")

plt.plot(entreno, label="entrenamiento (pasado)"); plt.plot(prueba, label="prueba (futuro)")
plt.axvline(prueba.index[0], color="k", ls="--"); plt.legend(); plt.show()

### 🔮 Ejercicio 5 — ¿Qué línea base gana?
Vamos a medir el error absoluto medio (MAE, Lección 4.1) de dos líneas base en los días de prueba:
- **Ingenua:** "hoy vendo lo mismo que ayer".
- **Estacional (7 días):** "hoy vendo lo mismo que el mismo día de la semana pasada".

¿Cuál tendrá **menos** error? Pista: recuerda el "serrucho" semanal.

✍️ *Mi predicción:*

In [ ]:
v = serie.values
real = v[corte:]
pred_ingenua    = v[corte - 1:-1]     # el valor de ayer
pred_estacional = v[corte - 7:-7]     # el valor de hace 7 días

print(f"MAE ingenua ('igual que ayer')          : {mean_absolute_error(real, pred_ingenua):.1f} unidades")
print(f"MAE estacional ('igual que hace 7 días'): {mean_absolute_error(real, pred_estacional):.1f} unidades")

<details><summary><b>👀 Ver respuesta</b></summary>

Gana la **estacional**: ≈9.7 contra ≈15.3 unidades. "Igual que ayer" falla sistemáticamente los viernes (ayer fue jueves, que vende menos) y los lunes (ayer fue domingo, que vende más). "Igual que el sábado pasado" ya trae incorporado el patrón semanal.

👉 Este ≈9.7 es ahora **la vara a superar**: un modelo que no baje de ahí no aporta nada.
</details>

### 📖 Un modelo de verdad: regresión lineal sobre ventanas
Ahora usamos las ventanas de la Sección 3 y una regresión lineal (Lección 5.1). El modelo aprende **cuánto pesa** cada día pasado para predecir el siguiente.

In [ ]:
def evaluar_ventana(tamano):
    X, y = crear_ventanas(serie, tamano)
    c = corte - tamano                          # la fila cuyo objetivo es el primer día de prueba
    modelo = LinearRegression().fit(X[:c], y[:c])
    return mean_absolute_error(y[c:], modelo.predict(X[c:])), modelo

error_14, modelo_14 = evaluar_ventana(14)
print(f"MAE regresión lineal con ventana de 14 días: {error_14:.1f} unidades")

### ✋ Ejercicio 6 — Lee los pesos
La celda de abajo dibuja el **peso** que la regresión le dio a cada día pasado (el 1 es "ayer", el 7 es "hace una semana").

¿Qué días pasados son los más importantes? ¿Tiene sentido?

In [ ]:
pesos = modelo_14.coef_[::-1]                  # reordenamos: primero "ayer"
plt.bar(range(1, 15), pesos, color=["tab:orange" if d in (7, 14) else "tab:blue" for d in range(1, 15)])
plt.xticks(range(1, 15)); plt.xlabel("hace cuántos días"); plt.ylabel("peso")
plt.title("¿En qué días del pasado se fija la regresión?"); plt.show()

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Los pesos más grandes están en **hace 7 y hace 14 días** (barras naranjas): el modelo descubrió solo el ciclo semanal, igual que la línea base estacional. Los demás días tienen pesos pequeños que ayudan a seguir la tendencia y a promediar el ruido.

Con eso el error baja a ≈8.7 unidades: le gana a la línea base estacional (≈9.7), aunque **no por mucho**. Así suele ser con series de tiempo: ganarle a una buena línea base cuesta.
</details>

---
## 5. 📖 ¿Qué pasa si mezclas el pasado con el futuro?

En la Lección 4.6 viste la validación cruzada **temporal** y en la 2.6 la **fuga de datos**. Con secuencias se juntan en el error más común:

> Usar `train_test_split` (que **baraja** las filas) con una serie de tiempo.

Al barajar, el modelo entrena con el martes y el jueves y se evalúa con el miércoles: ¡ya "vio el futuro"! En la vida real nunca tendrás el jueves cuando predices el miércoles.

### 🔮 Ejercicio 7 — La trampa del azar
Entrenaremos un **bosque aleatorio** (Módulo 6) con ventanas de 14 días de dos maneras:
- **Al azar:** 80 % de las filas para entrenar y 20 % para probar, barajadas.
- **Temporal:** los primeros 80 % de días para entrenar, los últimos 20 % para probar.

¿Cuál dará un error **más bajo**? ¿Cuál es el **honesto**?

✍️ *Mi predicción:*

In [ ]:
X14, y14 = crear_ventanas(serie, 14)
c14 = corte - 14

divisiones = {
    "al azar (❌ trampa)": train_test_split(X14, y14, test_size=0.2, random_state=0),
    "temporal (✅ honesta)": (X14[:c14], X14[c14:], y14[:c14], y14[c14:]),
}
for nombre, (X_tr, X_te, y_tr, y_te) in divisiones.items():
    bosque = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    print(f"Bosque aleatorio, división {nombre:<22}: MAE = {mean_absolute_error(y_te, bosque.predict(X_te)):.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Al azar:** MAE ≈ 8.2. ¡Parece mejor que todo lo anterior!
- **Temporal:** MAE ≈ 12.0. Peor incluso que la línea base estacional (≈9.7).

La división al azar es **optimista y falsa**: el bosque ve días de todo el periodo, incluidos los más recientes, así que nunca tiene que predecir ventas más altas de las que ya conoce. En la división temporal, la tienda sigue creciendo y el bosque **no puede extrapolar** (como el árbol de la mansión en la Lección 5.4): se queda "atascado" en los valores máximos que vio.

👉 Con secuencias, **siempre** valida respetando el tiempo.
</details>

### 📖 Validación cruzada temporal
Una sola división temporal puede tener suerte. `TimeSeriesSplit` hace varias, siempre **entrenando con el pasado y evaluando con el futuro inmediato**, con una ventana de entrenamiento que crece:

```
pliegue 1: [ENTRENO      ][prueba]
pliegue 2: [ENTRENO             ][prueba]
pliegue 3: [ENTRENO                    ][prueba]
```

In [ ]:
validacion = TimeSeriesSplit(n_splits=4)
for i, (idx_tr, idx_te) in enumerate(validacion.split(X14), start=1):
    modelo = LinearRegression().fit(X14[idx_tr], y14[idx_tr])
    error = mean_absolute_error(y14[idx_te], modelo.predict(X14[idx_te]))
    print(f"Pliegue {i}: entreno {len(idx_tr):>3} días → pruebo los {len(idx_te)} siguientes · MAE = {error:.1f}")

### ✋ Ejercicio 8
1. ¿Qué tienen en común todos los pliegues respecto a **dónde** están los datos de prueba?
2. Si un pliegue tuviera un error mucho mayor que los demás, ¿qué podría estar pasando en esa época?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. La prueba siempre está **justo después** del entrenamiento: nunca se usa el futuro para entrenar. Y el entrenamiento **crece** en cada pliegue.
2. Algo distinto pasa en esa época, o el modelo tenía poca historia. ¡Aquí ocurre! El **pliegue 2** tiene casi el doble de error (≈16 contra ≈8–9 de los demás). Ese modelo entrenó con menos de un año, así que solo vio una parte del ciclo anual y le tocó predecir justo la época que no conocía (otoño-invierno). Desde que tiene más de un año de historia (pliegues 3 y 4), el error vuelve a ≈9.

Esa es la gracia de la validación temporal: te deja ver **cómo cambia el error con el tiempo**, no solo un promedio. Con un solo corte no habrías visto que el modelo necesita al menos un año de datos.
</details>

---
## 6. 📖 La perilla principal: el tamaño de la ventana

¿Cuánto pasado le mostramos al modelo? Es el primer "hiperparámetro" de cualquier modelo de secuencias.

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Tamaño de ventana (*lags*, contexto) | Cuántos pasos pasados ve el modelo | Ve ciclos largos, pero hay menos filas y más ruido | Modelo más simple, pero "ciego" a los ciclos | Al menos **un ciclo completo** (7 días, 24 horas…) |
| Horizonte | Cuántos pasos hacia adelante predices | Problema más difícil, error mayor | Más fácil | El que pida el negocio |
| Tamaño de la prueba | Cuánto futuro reservas para evaluar | Evaluación más confiable, menos datos para entrenar | Al revés | Al menos un ciclo largo |

### ✏️ Ejercicio 9 — Busca la ventana
🔮 Antes de ejecutar: ¿qué crees que pasará con el error al pasar de una ventana de **3** días a una de **7**? ¿Y de 7 a 28?

In [ ]:
TAMANOS = [1, 3, 7, 14, 28]          # ✏️ prueba otros: 2, 6, 8, 60…
for tamano in TAMANOS:
    error, _ = evaluar_ventana(tamano)
    print(f"Ventana de {tamano:>2} días → MAE = {error:5.2f}  " + "█" * int(error))

✍️ *Mis observaciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

- De 1 a 3 días casi no mejora (≈15.1 → ≈14.5): con menos de una semana el modelo **no puede ver** el patrón semanal.
- De 3 a **7** días hay un salto enorme (≈14.5 → ≈9.0): ¡ya ve "el mismo día de la semana pasada"!
- De 7 a 28 la mejora es pequeña (≈9.0 → ≈8.5): más pasado ayuda a promediar el ruido, pero rinde cada vez menos.

Regla práctica: tu ventana debe cubrir **al menos un ciclo completo** del patrón más importante. Si pruebas ventanas como 6 u 8, verás que el 7 es "mágico" en esta serie.
</details>

### 📖 ¿Y cuando las secuencias tienen distinta longitud? *Padding*
Con texto no hay una "ventana" natural: cada frase mide lo que mide. Para meter varias frases en una misma tabla (un **lote**, *batch*) se **rellenan** las cortas con un símbolo especial hasta igualar a la más larga. A eso se le llama *padding*.

```
"hola"                  → [hola, <pad>, <pad>, <pad>]
"hola qué tal"          → [hola, qué,   tal,   <pad>]
"hola qué tal estás"    → [hola, qué,   tal,   estás]
```

### ✋ Ejercicio 10
Si en un lote hay una frase de 50 palabras y 31 frases de 5 palabras, ¿qué porcentaje de las casillas del lote es relleno? ¿Qué problema ves?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El lote mide 32 × 50 = 1,600 casillas. Con contenido real hay 50 + 31 × 5 = 205, así que el relleno es 1,395 / 1,600 ≈ **87 %**. Casi todo el cómputo se gasta en procesar relleno.

Por eso, en la práctica: (1) se agrupan frases de **longitud parecida** en el mismo lote, y (2) se usa una **máscara** que le dice al modelo "ignora el `<pad>`". Lo verás en acción con los modelos de las próximas lecciones.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Error bajísimo en validación, desastre en producción | Barajaste una serie de tiempo (fuga del futuro) | `TimeSeriesSplit` o corte temporal |
| Tu modelo no le gana a "mañana = hoy" | El modelo no aporta; o la serie es casi pura caminata aleatoria | Compara siempre con líneas base ingenuas y estacionales |
| Errores grandes siempre en ciertos días (lunes, festivos…) | La ventana no cubre el ciclo, o falta información del calendario | Ventana ≥ 1 ciclo; añadir día de la semana, festivos |
| Un árbol/bosque predice "plano" cuando la serie sube | Los árboles **no extrapolan** | Modelar la diferencia (cambio) en vez del nivel; o modelos lineales |
| Escalaste con media y desviación de **toda** la serie | Fuga: el escalador "vio" el futuro | Ajusta el escalador solo con el entrenamiento |
| Bolsa de palabras confunde frases con significado opuesto | Representación que ignora el orden | Modelos de secuencias (Lecciones 11.3–11.5) |

### 🔮 Ejercicio 11 — El escalador tramposo
Un compañero escala la serie de ventas así, **antes** de dividirla:
```python
serie_escalada = (serie - serie.mean()) / serie.std()
```
¿Qué información del futuro se le está "colando" al modelo?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

La **media** y la **desviación** se calcularon con los 730 días, incluidos los 146 de prueba. Como la tienda crece, la media de toda la serie es más alta que la del entrenamiento: el modelo "sabe" de forma indirecta que las ventas van a subir. Es una fuga sutil (Lección 2.6). Lo correcto: calcular media y desviación **solo con el entrenamiento** y aplicarlas después a la prueba.
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. En una **secuencia** el orden es información: texto, audio, series de tiempo, clics, ADN.
2. Las representaciones que ignoran el orden (como la bolsa de palabras) **pierden significado**.
3. Las **ventanas deslizantes** convierten una secuencia en una tabla: los últimos *n* pasos → el siguiente.
4. Antes de cualquier modelo, mide las **líneas base ingenuas** ("igual que ayer", "igual que la semana pasada"): son difíciles de vencer.
5. **Nunca barajes** una serie de tiempo para validar: entrena con el pasado y evalúa con el futuro (`TimeSeriesSplit`).

> 🔭 **Lo que viene:** las ventanas tienen un límite: el modelo solo ve *n* pasos y cada posición es una columna distinta, sin relación con las demás. En la Lección 11.3 conocerás las **redes recurrentes**, que leen la secuencia paso a paso y llevan una **memoria**. Pero antes, en la 11.2, resolveremos cómo convertir **palabras** en números.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Cuál de estos datos **no** es una secuencia en sentido estricto?
- a) Las lecturas por minuto de un sensor de temperatura
- b) Una reseña de un restaurante
- c) La tabla de características de 500 casas en venta
- d) Una grabación de voz

**2.** Con una serie de **100** pasos y ventanas de tamaño **10**, ¿cuántos ejemplos obtienes?
- a) 100 · b) 90 · c) 10 · d) 91

**3.** La demanda de un restaurante tiene un ciclo **semanal** fuerte. ¿Qué línea base esperas que sea más difícil de vencer?
- a) El promedio de todo el historial · b) "Igual que ayer" · c) "Igual que el mismo día de la semana pasada" · d) Siempre cero

**4.** Tu modelo de ventas tiene MAE = 4 en validación cruzada con `KFold(shuffle=True)` y MAE = 11 en producción. ¿Qué es lo **más probable**?
- a) Mala suerte · b) Fuga del futuro por barajar la serie · c) Le faltan épocas · d) La ventana es demasiado grande

**5.** ¿Para qué sirve el *padding*?
- a) Para quitar ruido del audio · b) Para igualar la longitud de secuencias de un mismo lote · c) Para aumentar los datos · d) Para escalar la serie

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** Las casas son filas independientes: puedes barajarlas sin perder nada.
2. **b)** 100 − 10 = 90.
3. **c)** La estacional ya incorpora el ciclo semanal. "Igual que ayer" falla cada cambio de día de la semana.
4. **b)** Barajar deja que el modelo entrene con días posteriores a los que evalúa. En producción ese "futuro" no existe.
5. **b)** Rellenar las secuencias cortas para que todas midan lo mismo dentro de un lote.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un analista dice: *"Mi modelo predice el precio de una acción para mañana con un error de solo 1 %"*. Antes de felicitarlo, ¿qué línea base le pedirías que te muestre y por qué?

**B. Caso de análisis 💼** Una cadena de cafeterías quiere predecir cuántos panes hornear **mañana** en cada tienda. Tiene 3 años de ventas diarias. Propón: (1) dos líneas base, (2) un tamaño de ventana razonable, (3) cómo dividirías los datos para validar, y (4) qué información **extra** (fuera de la serie) podría ayudar.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** La línea base **ingenua**: "el precio de mañana = el de hoy". En los precios de acciones, el cambio de un día suele ser de ~1 %, así que esa regla trivial ya logra un "error de 1 %". Lo que importa es si el modelo **le gana** a la ingenua, y casi nunca lo hace por mucho.

**B.** Una respuesta posible:
1. Líneas base: "igual que el mismo día de la semana pasada" y "promedio de los últimos 4 mismos días de la semana".
2. Ventana: al menos 7 días; probar 14 y 28. Si hay estacionalidad anual (fiestas), añadir el valor de hace 364 días.
3. Corte temporal: entrenar con los dos primeros años y validar con el tercero, o `TimeSeriesSplit` con varios pliegues; **nunca** barajar.
4. Información extra: día de la semana, festivos, clima, promociones, eventos cerca de la tienda, si la tienda estuvo cerrada.
</details>

### 9.3 🧩 Práctica integradora: la demanda eléctrica de una ciudad

La compañía eléctrica de una ciudad necesita predecir la demanda de la **próxima hora** para decidir cuántas plantas encender. Tienes 60 días de mediciones **por hora**.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng_e = np.random.default_rng(3)
horas = pd.date_range("2024-03-01", periods=24 * 60, freq="h")
hora_del_dia = horas.hour.values
demanda = (500 + 120 * np.sin(2 * np.pi * (hora_del_dia - 9) / 24)     # ciclo del día
           + 60 * np.exp(-((hora_del_dia - 20) ** 2) / 4)                # pico de la noche
           - 70 * (horas.day_of_week.values >= 5)                        # fines de semana: menos industria
           + rng_e.normal(0, 20, len(horas)))
electrica = pd.Series(demanda.round(1), index=horas, name="MW")

plt.figure(); electrica[:24 * 14].plot(title="Las primeras dos semanas (MW por hora)"); plt.show()
print(f"{len(electrica)} horas de datos")

✋ ¿Qué ciclos ves? ¿Cuántas horas dura cada uno?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Dos ciclos: uno **diario** (24 horas: baja de madrugada, sube de día, pico por la noche) y uno **semanal** (168 horas = 24 × 7: los sábados y domingos la demanda es más baja).
</details>

**Paso 2 🔮 — Tres líneas base.** Con el 20 % final como prueba, compararemos: "igual que hace 1 hora", "igual que hace 24 horas" y "igual que hace 168 horas (misma hora, semana pasada)". ¿Cuál crees que ganará?

✍️ *Mi apuesta:*

In [ ]:
ve = electrica.values
corte_e = int(len(ve) * 0.8)
real_e = ve[corte_e:]
for nombre, retraso in [("hace 1 hora", 1), ("hace 24 horas", 24), ("hace 168 horas", 168)]:
    print(f"Línea base 'igual que {nombre:<15}': MAE = {mean_absolute_error(real_e, ve[corte_e - retraso:-retraso]):.1f} MW")

<details><summary><b>👀 Ver respuesta</b></summary>

- Hace 1 hora: ≈28 MW · Hace 24 horas: ≈37 MW · Hace **168 horas: ≈22 MW** 🏆

¡Sorpresa! "Igual que ayer a la misma hora" es **peor** que "igual que hace una hora". ¿Por qué? Porque los lunes se comparan con el domingo y los sábados con el viernes: el ciclo semanal lo arruina. La línea base de 168 horas respeta **ambos** ciclos. La vara a superar es ≈22 MW.
</details>

**Paso 3 ✏️ — Regresión con ventanas.** Usa `crear_ventanas` y una regresión lineal con distintos tamaños de ventana. Elige `TAMANOS_E` para que al menos uno cubra el ciclo semanal.

In [ ]:
TAMANOS_E = [3, 24, 48, 168]     # ✏️

for tamano in TAMANOS_E:
    X_e, y_e = crear_ventanas(electrica, tamano)
    c = corte_e - tamano
    modelo_e = LinearRegression().fit(X_e[:c], y_e[:c])
    print(f"Ventana {tamano:>3} h → MAE = {mean_absolute_error(y_e[c:], modelo_e.predict(X_e[c:])):.1f} MW")

<details><summary><b>👀 Ver respuesta</b></summary>

Ventana 3 h ≈ 26.5 · 24 h ≈ 20.9 · 48 h ≈ 19.2 · **168 h ≈ 17.4 MW**. La regresión con una semana completa de contexto le gana a la mejor línea base (≈22) por unos 4–5 MW. Con solo 3 horas apenas mejora a "igual que hace 1 hora".
</details>

**Paso 4 ▶️ — Valida honestamente.** Un solo corte puede tener suerte. Haz validación cruzada temporal con 5 pliegues para la ventana de 24 horas y compárala, pliegue por pliegue, con la línea base de "hace 1 hora".

In [ ]:
X_e, y_e = crear_ventanas(electrica, 24)
for i, (idx_tr, idx_te) in enumerate(TimeSeriesSplit(n_splits=5).split(X_e), start=1):
    modelo_e = LinearRegression().fit(X_e[idx_tr], y_e[idx_tr])
    mae_modelo = mean_absolute_error(y_e[idx_te], modelo_e.predict(X_e[idx_te]))
    mae_ingenua = mean_absolute_error(y_e[idx_te], X_e[idx_te][:, -1])     # la última columna = hace 1 hora
    print(f"Pliegue {i}: modelo {mae_modelo:5.1f} MW · ingenua {mae_ingenua:5.1f} MW")

✋ ¿El modelo le gana a la ingenua en **todos** los pliegues? ¿Por qué importa eso más que el promedio?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Sí: en los 5 pliegues el modelo (≈20–23 MW) le gana a la ingenua (≈28–31 MW). Que gane **siempre**, en distintas épocas, da confianza de que la mejora es real y no suerte de un periodo concreto. Si ganara en promedio pero perdiera en la mitad de los pliegues, no te fiarías.
</details>

**Paso 5 ✋ — Tu recomendación.** Escribe en 3 frases qué le recomendarías a la compañía eléctrica: qué modelo, cuánto contexto y qué mejorarías después.

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

*"Recomiendo una regresión lineal que mira la última semana completa (168 horas): reduce el error de ~22 MW (la mejor regla simple, 'igual que hace una semana') a ~17 MW. La validamos respetando el tiempo, y le gana a las reglas simples en todos los periodos. Como siguiente paso, añadiría festivos y temperatura, y probaría modelos que leen la secuencia completa (LSTM, Lección 11.4)."*
</details>

---
## 📚 Para profundizar
- 📘 **Libro (gratis en línea):** Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (capítulos de líneas base y validación de series de tiempo; existe traducción al español).
- 📘 **Documentación:** scikit-learn, `TimeSeriesSplit` y el ejemplo *Time-related feature engineering*.
- 📺 **Video:** StatQuest, "Time Series" y "Bag of Words" (inglés, con subtítulos).

---
⬅️ **Anterior:** [10.7 Proyecto del módulo: Clasificador de vehículos para una empresa de logística 🚚✈️🚢](../m10-vision/10.7-proyecto-del-modulo.md) · ➡️ **Siguiente:** [11.2 Representar palabras: tokens y *embeddings* (el mapa de significados)](11.2-tokens-y-embeddings.ipynb)